## Формулировка задачи: «Лобстеры для банкетного меню»
Ресторан на побережье включает в банкетное меню блюда из свежих лобстеров, которых закупают каждое утро у местных рыбаков..

Экономические параметры:

Закупочная цена у поставщика, $c$ - $18 за кг

Цена продажи блюда посетителям, $p$ - $32 за кг

Цена реализации непроданных лобстеров (заморозка и продажа зоопарку на корм морским львам), $s$ - $9 за кг


Спрос:
$$
D∼N(μ=120 кг, σ=25 кг)
$$

Вопросы:

Какова будет мода, медиана, среднее значение, 10-й и 90-й перцентиль ежедневной прибыли при заданном уровне закупки $Q$?

Какая стратегия закупок минимизирует риск?

Сколько лобстеров следует закупать ежедневно? Обоснуйте решение.

## --------------------------------------------------------------------------------------------
Функция прибыли для фиксированного заказа $Q$ и реализовавшегося спроса $D$:

$$
Π(Q,D)=p⋅min(Q,D)+s⋅max(Q−D,0)−c⋅Q
$$

Шаги решения:

Сгенерировать выборку спроса размером $N=100000$ с помощью numpy.random.normal(mu, sigma, N) (с отсечением отрицательных значений, т.к. спрос не может быть отрицательным).

Для набора значений $Q$ (например, от μ−3σ до μ+3σ) рассчитать массив прибыли $Π(Q,D_i)$.

Для каждого $Q$ вычислить:

np.mean(profit) — среднее

np.median(profit) — медиана

мода — через гистограмму (np.histogram) или scipy.stats.mode

np.percentile(profit, 10) и np.percentile(profit, 90)


Построить графики (plotly/bokeh):

распределение прибыли (гистограмма) для нескольких $Q$;

зависимость средней прибыли, 10-го и 90-го перцентиля от $Q$ (для выбора оптимального заказа);

CDF прибыли — для оценки риска.


Теоретическая оптимальная точка заказа (критический коэффициент сервиса):
$$
Q^{∗} =μ + σ ⋅ Φ^{−1} \left(\frac{p−c}{p−s}\right)
$$
где $Φ^{−1}$ — обратная функция стандартного нормального распределения (scipy.stats.norm.ppf).
Это значение стоит сравнить с численным результатом симуляции.

## --------------------------------------------------------------------------------------------

### Работу выполнил
### студент: Мустаев Роберт Ильдарович
### группа: ВБ-9822-23

## 1. Модель

Параметры: $c=18$ \$/кг, $p=32$ \$/кг, $s=9$ \$/кг, спрос $D=\max(0,X)$, $X\sim N(120,\,25^2)$.

Подставим параметры в функцию прибыли:

$$
\Pi(Q,D)=32\min(Q,D)+9\max(Q-D,0)-18Q = 23\min(Q,D)-9Q .
$$

- если $D\ge Q$ (всё продано): $\Pi = 14Q$ — максимум прибыли при данном $Q$;
- если $D<Q$ (остались излишки): $\Pi = 23D-9Q$, прибыль падает на \$23 за каждый недостающий кг спроса.

Потери на 1 кг: нехватка (недополученная маржа) $c_u=p-c=14$, излишек $c_o=c-s=9$.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import plotly.graph_objects as go
from IPython.display import display


c, p, s = 18.0, 32.0, 9.0
mu, sigma = 120.0, 25.0
N = 100_000

np.random.seed(42)
demand = np.random.normal(mu, sigma, N)
demand = np.clip(demand, 0, None)

def profit(Q, D):
    """Прибыль за день при закупке Q и спросе D."""
    return p * np.minimum(Q, D) + s * np.maximum(Q - D, 0) - c * Q

print(f'Выборка спроса: среднее = {demand.mean():.2f} кг, ст. откл. = {demand.std(ddof=1):.2f} кг')
print(f'Отсечено отрицательных значений: {np.sum(demand == 0)}')

Выборка спроса: среднее = 120.02 кг, ст. откл. = 25.02 кг
Отсечено отрицательных значений: 0


## 2. Статистики прибыли для набора значений $Q$

Для сетки $Q$ от $\mu-3\sigma=45$ до $\mu+3\sigma=195$ кг (шаг 1 кг) считаем среднее, медиану, моду,
10-й и 90-й перцентили, а также показатели риска: стандартное отклонение и вероятность убытка.

**О моде.** При $D\ge Q$ прибыль равна ровно $14Q$, поэтому в распределении прибыли есть *атом*
(точка с положительной вероятностью $P(D\ge Q)$). Именно это значение — мода. Гистограммная оценка
(`np.histogram`) даёт интервал, содержащий $14Q$, пока вероятность атома $P(D\ge Q)$ больше массы любого
другого бина (в таблице — до $Q\approx165$ кг). При очень больших закупках (175+ кг) атом становится редким
(≈1%), и пик гистограммы смещается в непрерывную часть распределения — к прибыли $23\cdot120-9Q$
при типичном спросе. Оценка гистограммой зависит от числа бинов, поэтому приводим обе.

In [2]:
def hist_mode(values, bins=100):
    """Мода через гистограмму: середина самого заполненного интервала."""
    counts, edges = np.histogram(values, bins=bins)
    j = counts.argmax()
    return (edges[j] + edges[j + 1]) / 2

def profit_stats(Q):
    v = profit(Q, demand)
    return {
        'Q, кг': Q,
        'Среднее, $': v.mean(),
        'Медиана, $': np.median(v),
        'Мода (гистограмма), $': hist_mode(v),
        'Мода (атом 14Q), $': 14 * Q,
        'P10, $': np.percentile(v, 10),
        'P90, $': np.percentile(v, 90),
        'Ст. откл., $': v.std(ddof=1),
        'P(убыток), %': 100 * np.mean(v < 0),
        'P(всё продано), %': 100 * np.mean(demand >= Q),
    }

q_grid = np.arange(mu - 3 * sigma, mu + 3 * sigma + 1, 1.0)
grid = pd.DataFrame([profit_stats(Q) for Q in q_grid])

display(grid[grid['Q, кг'] % 10 == 5].round(2).reset_index(drop=True))

,"Q, кг","Среднее, $","Медиана, $","Мода (гистограмма), $","Мода (атом 14Q), $","P10, $","P90, $","Ст. откл., $","P(убыток), %","P(всё продано), %"
0,45.0,629.78,630.00,625.79,630.0,630.00,630.00,8.42,0.00,99.87
1,55.0,769.16,770.00,764.64,770.0,770.00,770.00,16.88,0.00,99.52
2,65.0,907.15,910.00,903.49,910.0,910.00,910.00,32.35,0.01,98.60
3,75.0,1041.75,1050.00,1042.34,1050.0,1050.00,1050.00,57.82,0.02,96.40
4,85.0,1168.93,1190.00,1181.19,1190.0,1190.00,1190.00,96.58,0.02,91.98
5,95.0,1282.10,1330.00,1320.04,1330.0,1168.23,1330.00,150.49,0.04,84.15
6,105.0,1373.00,1470.00,1458.89,1470.0,1078.23,1470.00,218.72,0.08,72.58
7,115.0,1433.49,1610.00,1597.74,1610.0,988.23,1610.00,296.16,0.13,57.93
8,125.0,1458.61,1636.52,1736.59,1750.0,898.23,1750.00,374.39,0.23,42.05
9,135.0,1448.08,1546.52,1875.44,1890.0,808.23,1890.00,444.02,0.35,27.41


## 3. Ответ на вопрос 1: статистики при заданном $Q$

Задайте объём закупки в переменной `Q_user` — ниже выводятся все требуемые характеристики.
Для сравнения добавлены ключевые стратегии, найденные далее.

Поскольку прибыль монотонно не убывает по спросу, её квантили выражаются через квантили спроса:
$\Pi_\alpha = 23\min(Q, D_\alpha) - 9Q$, где $D_{0.1}=87.96$, $D_{0.5}=120$, $D_{0.9}=152.04$ кг.

In [3]:
Q_user = 120.0

def exact_mean(Q):
    """Теоретическое матожидание прибыли (с учётом отсечения D = max(0, X))."""
    Q = np.asarray(Q, dtype=float)
    def pos_part(a):                       # E[(X - a)+]
        z = (mu - a) / sigma
        return (mu - a) * norm.cdf(z) + sigma * norm.pdf(z)
    return (p - s) * (pos_part(0) - pos_part(Q)) - (c - s) * Q

v = profit(Q_user, demand)
print(f'Закупка Q = {Q_user:.0f} кг')
print(f'  Среднее            : ${v.mean():9.2f}   (теория: ${exact_mean(Q_user):.2f})')
print(f'  Медиана            : ${np.median(v):9.2f}   (теория: ${23*min(Q_user, mu) - 9*Q_user:.2f})')
print(f'  Мода               : ${14*Q_user:9.2f}   (гистограмма: ${hist_mode(v):.2f})')
print(f'  10-й перцентиль    : ${np.percentile(v, 10):9.2f}')
print(f'  90-й перцентиль    : ${np.percentile(v, 90):9.2f}')

Закупка Q = 120 кг
  Среднее            : $  1450.63   (теория: $1450.61)
  Медиана            : $  1680.00   (теория: $1680.00)
  Мода               : $  1680.00   (гистограмма: $1667.16)
  10-й перцентиль    : $   943.23
  90-й перцентиль    : $  1680.00


## 4. Теоретический оптимум и сравнение с симуляцией

Критический коэффициент сервиса:

$$
\frac{p-c}{p-s}=\frac{14}{23}\approx 0.6087,\qquad
Q^{*}=\mu+\sigma\,\Phi^{-1}(0.6087)=120+25\cdot 0.2759\approx 126.9\ \text{кг}.
$$

Коэффициент больше 0.5, так как недопоставка (\$14/кг) дороже излишка (\$9/кг) — поэтому выгодно
закупать **больше среднего спроса**.

In [4]:
service = (p - c) / (p - s)
q_star = mu + sigma * norm.ppf(service)

fine = np.arange(100, 150.01, 0.1)
sim_means = np.array([profit(Q, demand).mean() for Q in fine])
q_sim = fine[sim_means.argmax()]

q_int = max([np.floor(q_star), np.ceil(q_star)], key=exact_mean)

print(f'Критический коэффициент (p-c)/(p-s) = {service:.4f}')
print(f'Теоретический оптимум Q*          = {q_star:.2f} кг, E[прибыль] = ${exact_mean(q_star):.2f}')
print(f'Оптимум по симуляции              = {q_sim:.1f} кг, средняя прибыль = ${sim_means.max():.2f}')
print(f'Расхождение                       = {abs(q_sim - q_star):.2f} кг')
print(f'Лучший целый объём                = {q_int:.0f} кг, E[прибыль] = ${exact_mean(q_int):.2f}')
print(f'Плато оптимума: при Q = 120 и 135 кг средняя прибыль ниже лишь на '
      f'${exact_mean(q_star) - exact_mean(120):.0f} и ${exact_mean(q_star) - exact_mean(135):.0f}')

Критический коэффициент (p-c)/(p-s) = 0.6087
Теоретический оптимум Q*          = 126.90 кг, E[прибыль] = $1459.18
Оптимум по симуляции              = 126.9 кг, средняя прибыль = $1459.24
Расхождение                       = 0.00 кг
Лучший целый объём                = 127 кг, E[прибыль] = $1459.17
Плато оптимума: при Q = 120 и 135 кг средняя прибыль ниже лишь на $9 и $11


## 5. Графики

In [5]:
q_safe = norm.ppf(0.10, loc=mu, scale=sigma)
show_Q = [q_safe, 110.0, q_star, 150.0]

# 5.1
fig = go.Figure()
for Q in show_Q:
    fig.add_trace(go.Histogram(x=profit(Q, demand), name=f'Q = {Q:.0f} кг',
                               histnorm='probability', xbins=dict(size=40), opacity=0.55))
fig.update_layout(title='Распределение ежедневной прибыли при разных Q', barmode='overlay',
                  xaxis_title='Прибыль, $', yaxis_title='Доля дней', template='plotly_white')
fig.show()

# 5.2
fig = go.Figure()
fig.add_trace(go.Scatter(x=grid['Q, кг'], y=grid['P90, $'], name='90-й перцентиль'))
fig.add_trace(go.Scatter(x=grid['Q, кг'], y=grid['Среднее, $'], name='Среднее (симуляция)'))
fig.add_trace(go.Scatter(x=grid['Q, кг'], y=exact_mean(grid['Q, кг']), name='Среднее (теория)',
                         line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=grid['Q, кг'], y=grid['P10, $'], name='10-й перцентиль'))
fig.add_vline(x=q_star, line_dash='dot', annotation_text=f'Q* = {q_star:.1f}')
fig.add_vline(x=q_safe, line_dash='dot', annotation_text=f'max P10: {q_safe:.1f}',
              annotation_position='top left')
fig.update_layout(title='Прибыль в зависимости от объёма закупки', xaxis_title='Закупка Q, кг',
                  yaxis_title='Прибыль, $', template='plotly_white')
fig.show()

# 5.3
fig = go.Figure()
for Q in show_Q:
    x = np.sort(profit(Q, demand))
    y = np.arange(1, N + 1) / N
    step = max(1, N // 2000)
    fig.add_trace(go.Scatter(x=np.r_[x[::step], x[-1]], y=np.r_[y[::step], 1.0],
                             mode='lines', line_shape='hv', name=f'Q = {Q:.0f} кг'))
fig.add_hline(y=0.10, line_dash='dot', annotation_text='10%')
fig.add_vline(x=0, line_dash='dot')
fig.update_layout(title='CDF прибыли: P(прибыль ≤ x)', xaxis_title='Прибыль, $',
                  yaxis_title='Накопленная вероятность', template='plotly_white')
fig.show()

## 6. Ответ на вопрос 2: стратегия, минимизирующая риск

Сравним стратегии по нескольким мерам риска:
- **стандартное отклонение** прибыли (разброс);
- **вероятность убытка** $P(\Pi<0)=\Phi\left(\frac{9Q/23-\mu}{\sigma}\right)$;
- **10-й перцентиль** (VaR 10%) — прибыль, которую ресторан получает не менее чем в 90% дней;
- **CVaR 10%** — средняя прибыль в 10% худших дней.

In [6]:
def risk_row(name, Q):
    v = profit(Q, demand)
    worst = np.sort(v)[:len(v) // 10]
    return {'Стратегия': name, 'Q, кг': round(Q, 1),
            'Среднее, $': v.mean(), 'Ст. откл., $': v.std(ddof=1),
            'P10, $': np.percentile(v, 10), 'CVaR 10%, $': worst.mean(),
            'P(убыток), %': 100 * np.mean(v < 0),
            'Излишки в среднем, кг': np.maximum(Q - demand, 0).mean()}

risk = pd.DataFrame([
    risk_row('Минимум риска (max P10)', q_safe),
    risk_row('Компромисс', 110.0),
    risk_row('Закупка = средний спрос', mu),
    risk_row('Максимум средней прибыли', q_int),
    risk_row('Перезакупка', 150.0),
]).round(2)
display(risk)

p10_best = grid.loc[grid['P10, $'].idxmax(), 'Q, кг']
print(f'Максимум P10 на сетке: Q = {p10_best:.0f} кг (теория: D_0.1 = {q_safe:.2f} кг)')
print(f'Цена осторожности: средняя прибыль ниже на ${exact_mean(q_int) - exact_mean(q_safe):.0f} в день '
      f'({100*(1 - exact_mean(q_safe)/exact_mean(q_int)):.1f}%)')

,Стратегия,"Q, кг","Среднее, $","Ст. откл., $","P10, $","CVaR 10%, $","P(убыток), %","Излишки в среднем, кг"
0,Минимум риска (max P10),88.0,1204.26,110.92,1231.46,959.53,0.03,1.18
1,Компромисс,110.0,1407.46,256.76,1033.23,761.18,0.10,5.76
2,Закупка = средний спрос,120.0,1450.63,335.80,943.23,671.18,0.18,9.97
3,Максимум средней прибыли,127.0,1459.24,389.27,880.23,608.18,0.25,13.86
4,Перезакупка,150.0,1378.17,519.37,673.23,401.18,0.74,31.38


Максимум P10 на сетке: Q = 88 кг (теория: D_0.1 = 87.96 кг)
Цена осторожности: средняя прибыль ниже на $255 в день (17.5%)


**Вывод по вопросу 2.** Все меры риска (разброс, вероятность убытка, излишки) **растут вместе с $Q$**:
чем больше закупка, тем больше денег «под риском» в дни слабого спроса. Тривиальная стратегия $Q=0$
убирает риск полностью, но и прибыль нулевая, поэтому разумный критерий — **максимизировать гарантированную
прибыль, т. е. 10-й перцентиль**.

Риск минимизирует закупка на уровне 10-го перцентиля спроса:

$$Q_{\text{safe}} = \mu + \sigma\,\Phi^{-1}(0.1) = 120 - 1.2816\cdot25 \approx 88\ \text{кг}.$$

При ней в 90% дней весь товар распродаётся, прибыль почти постоянна (≈\$1231), её разброс минимален среди
разумных вариантов, а вероятность убытка ≈ 0.03%. Плата за это — средняя прибыль примерно на \$255 в день
(≈17%) ниже максимальной, и в 90% дней ресторану не хватает лобстеров (теряются гости).

## 7. Ответ на вопрос 3: сколько закупать

In [7]:
v_rec = profit(q_int, demand)
print(f'Рекомендуемая закупка: {q_int:.0f} кг в день')
print(f'  ожидаемая прибыль       : ${exact_mean(q_int):.2f} в день  (≈ ${365*exact_mean(q_int):,.0f} в год)')
print(f'  медиана / мода          : ${np.median(v_rec):.2f} / ${14*q_int:.2f}')
print(f'  P10 / P90               : ${np.percentile(v_rec, 10):.2f} / ${np.percentile(v_rec, 90):.2f}')
print(f'  вероятность убытка      : {100*np.mean(v_rec < 0):.2f}%')
print(f'  весь товар продан       : {100*np.mean(demand >= q_int):.1f}% дней')
print(f'Выигрыш против осторожной стратегии (88 кг): '
      f'${exact_mean(q_int) - exact_mean(q_safe):.0f} в день, ≈ ${365*(exact_mean(q_int) - exact_mean(q_safe)):,.0f} в год')

Рекомендуемая закупка: 127 кг в день
  ожидаемая прибыль       : $1459.17 в день  (≈ $532,599 в год)
  медиана / мода          : $1618.52 / $1778.00
  P10 / P90               : $880.23 / $1778.00
  вероятность убытка      : 0.25%
  весь товар продан       : 39.0% дней
Выигрыш против осторожной стратегии (88 кг): $255 в день, ≈ $93,053 в год


**Рекомендация: закупать 127 кг лобстеров ежедневно.**

Обоснование:
1. **Это максимум ожидаемой прибыли.** Теоретический оптимум по критическому коэффициенту
   $Q^*=126.9$ кг совпадает с результатом симуляции (расхождение меньше 0.5 кг); из целых значений лучше 127 кг.
2. **Асимметрия потерь.** Недостающий килограмм стоит ресторану \$14 упущенной маржи, лишний — только \$9
   (остатки уходят зоопарку). Поэтому закупать выгодно немного больше среднего спроса в 120 кг.
3. **Риск при этом мал.** Убыток бывает лишь в ≈0.24% дней (примерно раз в год), даже в 10% худших дней прибыль
   положительна (P10 ≈ \$880), а весь товар распродаётся примерно в 39% дней.
4. **Осторожная стратегия (88 кг) дорога:** она снижает разброс, но теряет ≈\$255 в день (≈\$93 тыс. в год)
   и в 9 днях из 10 оставляет гостей банкетов без блюда.
5. **Оптимум пологий:** при отклонении закупки на ±7 кг средняя прибыль падает лишь примерно на \$10,
   поэтому ошибки в оценке спроса не критичны.

Если ресторан крайне не склонен к риску, можно выбрать компромисс ≈110 кг (средняя прибыль ниже на ≈\$50,
разброс заметно меньше), но для коммерческого ресторана с ежедневными повторяющимися закупками рациональнее
ориентироваться на среднюю прибыль: за год случайные колебания усредняются.

## 8. Проверки

In [8]:
assert np.all(profit(0, demand) == 0)
assert np.isclose(profit(100, 0), -900)
assert np.isclose(profit(100, 150), 1400)
assert np.isclose(norm.cdf(q_star, mu, sigma), service)
assert abs(q_sim - q_star) < 1.0
v = profit(q_star, demand)
assert abs(v.mean() - exact_mean(q_star)) < 5 * v.std(ddof=1) / np.sqrt(N)
assert exact_mean(q_star) > max(exact_mean(q_star - 1), exact_mean(q_star + 1))
assert abs(p10_best - q_safe) <= 1.0
print('Все проверки пройдены.')

Все проверки пройдены.


## Итоги

| Вопрос | Ответ |
|---|---|
| Статистики прибыли при заданном $Q$ | Среднее — `exact_mean(Q)`; медиана $23\min(Q,120)-9Q$; мода $14Q$ (атом «всё продано»); P10 $23\min(Q,87.96)-9Q$; P90 $23\min(Q,152.04)-9Q$ (раздел 3, таблица в разделе 2) |
| Стратегия минимального риска | Закупка ≈ **88 кг** (10-й перцентиль спроса): максимальная гарантированная прибыль ≈ \$1231 в 90% дней, минимальный разброс |
| Рекомендуемая закупка | **127 кг в день**: максимум ожидаемой прибыли ≈ \$1459/день, риск убытка ≈ 0.24% |